# 한국 근대사 교육 챗봇 (QWEN 2.5 기반)

이 노트북은 **Qwen2.5 모델**과 **한국근대사 학습 자료**(`korean_modern_history_chatbot_ready.txt`)를 결합하여  
역사 교육용 챗봇 로직(`history_logic.py`)을 생성합니다.

## 전체 흐름
1. 필요 라이브러리 설치 및 임포트
2. 한국근대사 텍스트 데이터 로드 및 섹션 분할
3. QWEN 2.5 모델 로드
4. 키워드 기반 RAG(검색 증강 생성) 함수 구현
5. 챗봇 응답 함수 구현 및 테스트
6. `history_logic.py` 파일로 내보내기 → NewLearn `app.py`에 병합

## 1. 환경 준비

### 설치 패키지
- `transformers`: QWEN 모델 로드 및 추론
- `torch`: GPU/CPU 텐서 연산
- `accelerate`: 모델 최적화 로딩 지원

In [ ]:
# 필요한 패키지를 설치합니다.
# transformers: Hugging Face 모델 로드
# accelerate: 대형 모델 최적화 로딩
!pip install -q transformers torch accelerate

## 2. 라이브러리 임포트

In [11]:
import os
import re
import gc
import torch
from transformers import pipeline
from transformers.utils import logging as hf_logging

# 불필요한 progress bar 출력을 비활성화합니다.
hf_logging.disable_progress_bar()

# GPU 메모리 최적화 설정
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU 사용 가능: {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("CPU 모드로 실행합니다. (GPU 없음)")

CPU 모드로 실행합니다. (GPU 없음)


## 3. 한국근대사 학습 데이터 로드

### 처리 방식
- `##` 헤더를 기준으로 섹션을 분할합니다.
- 각 섹션에서 **핵심 포인트**, **주요 키워드**, **예상 질문**을 추출합니다.
- 사용자 질문과 섹션의 키워드를 비교해 가장 관련 있는 내용을 찾습니다(RAG).

In [12]:
# 학습 데이터 파일 경로
# 같은 폴더에 있는 경우 파일명만, 다른 경로면 절대 경로로 변경하세요.
DATA_FILE = "korean_modern_history_chatbot_ready.txt"

def load_history_data(file_path: str) -> list[dict]:
    """
    한국근대사 텍스트 파일을 읽어 섹션 단위 리스트로 반환합니다.
    
    Returns:
        list of dict: [
            {
                'title': 섹션 제목,
                'content': 전체 내용,
                'keywords': [키워드 리스트],
                'questions': [예상 질문 리스트]
            }, ...
        ]
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        raw_text = f.read()
    
    # '###' 소제목 기준으로 분할 (큰 제목은 건너뜀)
    # '##' 이후 '###' 나오기 전까지를 서론으로 처리
    sections = re.split(r'(?=###)', raw_text)
    
    parsed = []
    for section in sections:
        section = section.strip()
        if not section or not section.startswith('#'):
            continue
        
        # 제목 추출 (### 또는 ## 이후 첫 줄)
        title_match = re.match(r'#{2,3}\s+(.+)', section)
        title = title_match.group(1).strip() if title_match else "(제목 없음)"
        
        # 키워드 추출 ("주요 키워드:" 이후 내용)
        keyword_match = re.search(r'주요 키워드[:\s]+(.+)', section)
        if keyword_match:
            keywords = [kw.strip() for kw in keyword_match.group(1).split(',')]
        else:
            keywords = []
        
        # 예상 질문 추출 ("예상 질문:" 이후 내용)
        question_match = re.search(r'예상 질문[:\s]+(.+)', section)
        if question_match:
            questions = [q.strip() for q in question_match.group(1).split('/')]
        else:
            questions = []
        
        parsed.append({
            'title': title,
            'content': section,
            'keywords': keywords,
            'questions': questions
        })
    
    return parsed


# 데이터 로드 및 확인
history_sections = load_history_data(DATA_FILE)
print(f"총 {len(history_sections)}개 섹션 로드 완료\n")
for i, sec in enumerate(history_sections[:3]):
    print(f"[{i+1}] 제목: {sec['title']}")
    print(f"     키워드: {sec['keywords'][:5]}")
    print()

총 21개 섹션 로드 완료

[1] 제목: (제목 없음)
     키워드: []

[2] 제목: 사대교린 질서의 동요와 조선 내부 위기 (1860년대 전후)
     키워드: ['사대교린', '아편전쟁', '양무운동', '농민층 분화', '환곡']

[3] 제목: 흥선대원군의 집권과 부국강병 정책 (1863-1873)
     키워드: ['흥선대원군', '부국강병', '비변사 폐지', '경복궁 중건', '서원 철폐']



## 4. QWEN 2.5 모델 로드

### 모델 선택
- `Qwen/Qwen2.5-1.5B-Instruct`: 약 1.5B 파라미터, GPU 메모리 ~4GB 필요
- `Qwen/Qwen2.5-3B-Instruct`: 약 3B 파라미터, GPU 메모리 ~8GB 필요 (성능 ↑)
- `Qwen/Qwen2.5-7B-Instruct`: 약 7B 파라미터, GPU 메모리 ~16GB 필요 (성능 ↑↑)

GPU 메모리에 맞게 `MODEL_ID`를 조정하세요.

### 작동 방식
- `pipeline("text-generation")`으로 간편하게 모델을 래핑합니다.
- `device=0`이면 GPU, `device=-1`이면 CPU를 사용합니다.

In [13]:
# 사용할 QWEN 모델 ID
# GPU 메모리가 충분하면 3B나 7B로 변경 가능합니다.
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# GPU 사용 여부에 따라 device 설정
DEVICE = 0 if torch.cuda.is_available() else -1

print(f"모델 로딩 중: {MODEL_ID}")
print(f"디바이스: {'GPU' if DEVICE == 0 else 'CPU'}")

# Hugging Face pipeline으로 모델 로드
# trust_remote_code=True: 모델 자체 커스텀 코드를 신뢰하여 실행
qwen_pipe = pipeline(
    task="text-generation",
    model=MODEL_ID,
    device=DEVICE,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == 0 else torch.float32  # GPU면 float16(메모리 절약)
)

print("\n모델 로드 완료!")

모델 로딩 중: Qwen/Qwen2.5-3B-Instruct
디바이스: CPU

모델 로드 완료!


## 5. RAG 검색 함수 구현

### RAG(Retrieval-Augmented Generation)란?
- 사용자 질문과 가장 관련 있는 **학습 자료 섹션을 먼저 검색**합니다.
- 검색된 내용을 **프롬프트에 포함**시켜 모델이 정확한 정보를 바탕으로 답변하게 합니다.
- 이 방법으로 모델의 '환각(Hallucination)' 현상을 줄이고 역사 사실 기반 응답을 유도합니다.

### 검색 방식
- 사용자 질문에서 형태소를 추출하고 섹션의 키워드·제목과 **단어 일치 점수**를 계산합니다.
- 점수가 높은 상위 2개 섹션의 내용을 컨텍스트로 사용합니다.

In [14]:
def retrieve_context(user_query: str, sections: list[dict], top_k: int = 2) -> str:
    """
    사용자 질문과 가장 관련 있는 섹션을 검색하여 컨텍스트 문자열로 반환합니다.
    
    Args:
        user_query: 사용자 질문
        sections: load_history_data()가 반환한 섹션 리스트
        top_k: 반환할 상위 섹션 수
    
    Returns:
        str: 컨텍스트 문자열 (모델 프롬프트에 삽입)
    """
    # 쿼리를 단어 단위로 분할 (2글자 이상 단어만 사용)
    query_tokens = set(re.findall(r'[가-힣a-zA-Z0-9]{2,}', user_query))
    
    scores = []
    for sec in sections:
        score = 0
        
        # 제목과 키워드에서 단어 매칭 점수 계산
        title_tokens = set(re.findall(r'[가-힣a-zA-Z0-9]{2,}', sec['title']))
        kw_text = ' '.join(sec['keywords'])
        kw_tokens = set(re.findall(r'[가-힣a-zA-Z0-9]{2,}', kw_text))
        q_text = ' '.join(sec['questions'])
        q_tokens = set(re.findall(r'[가-힣a-zA-Z0-9]{2,}', q_text))
        
        # 제목 일치: 가중치 3 / 키워드 일치: 가중치 2 / 예상질문 일치: 가중치 1
        score += len(query_tokens & title_tokens) * 3
        score += len(query_tokens & kw_tokens) * 2
        score += len(query_tokens & q_tokens) * 1
        
        scores.append((score, sec))
    
    # 점수 내림차순 정렬 후 상위 top_k 선택
    scores.sort(key=lambda x: x[0], reverse=True)
    top_sections = [sec for _, sec in scores[:top_k] if _ > 0]
    
    if not top_sections:
        # 관련 섹션이 없으면 전체 요약 제공
        return "이 챗봇은 한국 근대사(1860년대~1945년) 전반을 다룹니다."
    
    # 컨텍스트 조합 (각 섹션의 핵심 내용 추출)
    context_parts = []
    for sec in top_sections:
        # 섹션 내용에서 '핵심 포인트' 이후 '주요 키워드' 이전 내용 추출
        core_match = re.search(r'핵심 포인트:\s*([\s\S]+?)(?=주요 키워드|예상 질문|$)', sec['content'])
        if core_match:
            core_text = core_match.group(1).strip()
        else:
            # 핵심 포인트가 없으면 전체 내용의 처음 300자 사용
            core_text = sec['content'][:300]
        
        context_parts.append(f"[{sec['title']}]\n{core_text}")
    
    return "\n\n".join(context_parts)


# 테스트: 검색 함수 동작 확인
test_query = "갑신정변의 원인은 무엇인가요?"
context = retrieve_context(test_query, history_sections)
print(f"질문: {test_query}")
print(f"\n검색된 컨텍스트:\n{context[:500]}...")

질문: 갑신정변의 원인은 무엇인가요?

검색된 컨텍스트:
[문명개화론 수용과 갑신정변의 성격 (1884)]
- 개화파는 문명개화론을 통해 근대 국가상을 구상했다.
- 갑신정변은 권력 장악을 통한 급진 개혁 시도였다.
- 정변의 개혁안에는 신분 질서 변동과 국가기구 개편 요소가 있었다.
- 일본 의존과 사회 기반 부족이 치명적 약점이었다.
- 실패 이후 조선 내 개혁 구도는 더욱 복잡해졌다.

[임오군란 이후 국제 정세와 갑신정변의 배경 (1882-1884)]
- 임오군란은 청의 내정 간섭을 강화시키는 계기였다.
- 일본은 제물포조약을 통해 군사적·외교적 발판을 유지했다.
- 청프전쟁은 조선 정세에 새로운 균열을 만들었다.
- 급진개화파는 국제 환경 변화를 정변의 기회로 판단했다.
- 정변 배경에는 국내 개혁 의지와 외세 의존의 모순이 함께 존재했다....


## 6. 챗봇 응답 생성 함수

### 프롬프트 설계
- **system**: 챗봇의 역할과 응답 스타일을 정의합니다.
- **user**: 검색된 컨텍스트 + 대화 이력 + 현재 질문을 결합합니다.

### 생성 파라미터
- `max_new_tokens=512`: 최대 512 토큰 생성 (너무 길면 줄여도 됨)
- `temperature=0.3`: 낮을수록 더 일관된(안정적인) 응답
- `do_sample=True`: 샘플링 방식으로 다양한 표현 허용
- `repetition_penalty=1.2`: 같은 내용 반복 패널티

In [15]:
# 시스템 프롬프트: 챗봇의 역할과 규칙을 정의합니다.
SYSTEM_PROMPT = """당신은 한국 근대사 전문 교육 챗봇입니다.
아래 [참고 자료]를 바탕으로 학생의 질문에 친절하고 정확하게 답변하세요.

응답 규칙:
1. 반드시 한국어(한글)로만 답변합니다.
2. 한자를 절대 출력하지 않습니다. 괄호 안에도 한자 금지입니다.
3. [참고 자료]에 있는 내용만을 바탕으로 답변합니다. 자료에 없는 내용은 추가하지 않습니다.
4. 모르는 내용은 '이 부분은 제 자료에 없습니다'라고 말합니다.
5. 중학생이 이해하기 쉽게 기 - 승 - 전 - 결 순서로 설명합니다.
6. 어려운 역사 용어는 바로 뒤에 쉬운 말로 풀어 설명합니다. 예: "통리기무아문 — 외교·군사 담당 기관"
7. 선생님이 학생에게 이야기하는 말투로 작성합니다.
8. 4문장 이상으로 답변합니다."""


def remove_hanja(text: str) -> str:
    """생성된 텍스트에서 한자(CJK)를 제거합니다."""
    return re.sub(r'[一-鿿㐀-䶿豈-﫿]+', '', text)


def generate_history_response(
    user_query: str,
    chat_history: list[dict] = None,
    sections: list[dict] = None,
    pipe=None
) -> str:
    """
    QWEN 모델로 한국 근대사 질문에 대한 답변을 생성합니다.
    
    Args:
        user_query: 현재 사용자 질문
        chat_history: 이전 대화 이력 [{role, content}, ...] 형식
        sections: 역사 데이터 섹션 리스트
        pipe: transformers pipeline 객체
    
    Returns:
        str: 모델이 생성한 답변 텍스트
    """
    if chat_history is None:
        chat_history = []
    if sections is None:
        sections = []
    
    # 1. RAG: 사용자 질문과 관련된 역사 자료 검색
    context = retrieve_context(user_query, sections, top_k=2)
    
    # 2. 메시지 구성
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]
    
    # 이전 대화 이력 추가 (최근 4턴만 포함 — 컨텍스트 길이 절약)
    recent_history = chat_history[-8:] if len(chat_history) > 8 else chat_history
    for turn in recent_history:
        role = "assistant" if turn.get("role") == "bot" else "user"
        # HTML 태그 제거 (app.py에서 HTML로 렌더링하므로)
        content = re.sub(r'<[^>]+>', '', turn.get("content", ""))
        messages.append({"role": role, "content": content})
    
    # 현재 질문 (컨텍스트 포함)
    user_content = f"[참고 자료]\n{context}\n\n[질문]\n{user_query}"
    messages.append({"role": "user", "content": user_content})
    
    # 3. 모델 추론
    outputs = pipe(
        messages,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        repetition_penalty=1.2,
        return_full_text=False  # 입력 프롬프트 제외하고 생성된 부분만 반환
    )
    
    # 4. 결과 텍스트 추출
    generated = outputs[0]['generated_text']
    
    # 리스트 형식으로 반환되는 경우 처리
    if isinstance(generated, list):
        # 마지막 assistant 메시지 추출
        for msg in reversed(generated):
            if isinstance(msg, dict) and msg.get('role') == 'assistant':
                return remove_hanja(msg.get('content', '').strip())
        return remove_hanja(str(generated[-1]).strip())
    
    return remove_hanja(str(generated).strip())


print("챗봇 응답 함수 정의 완료")

챗봇 응답 함수 정의 완료


## 7. 챗봇 테스트

다양한 질문으로 챗봇 응답을 확인합니다.

In [16]:
# 테스트 질문 목록
test_questions = [
    "흥선대원군의 핵심 정책은 무엇이었나요?",
    "갑신정변이 실패한 이유는 무엇인가요?",
    "1894년 농민전쟁의 원인과 성격은 무엇인가요?",
]

chat_history = []  # 대화 이력 초기화

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"질문: {q}")
    print("-" * 60)
    
    response = generate_history_response(
        user_query=q,
        chat_history=chat_history,
        sections=history_sections,
        pipe=qwen_pipe
    )
    
    print(f"답변:\n{response}")
    
    # 대화 이력에 추가 (다음 질문에서 맥락 유지)
    chat_history.append({"role": "user", "content": q})
    chat_history.append({"role": "bot", "content": response})

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



질문: 흥선대원군의 핵심 정책은 무엇이었나요?
------------------------------------------------------------


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


답변:
흥선대원군의 주요 정책 중 하나인 ‘부국강병’이었습니다. 이정책은 나라를 더욱 번성시키고 군사를 더 잘 유지하자는 취지였습니다. 하지만 그때 당시에는 서양과 직접적인 접촉보다는 자신들의 문화와 관습을 보존하면서 국력을 증진시키려 노력했습니다. 이렇게 하여 왕실 권력 강화, 사회 구조 변화 등을 통해 새로운 시대로 나아가는 계기를 마련했죠.

질문: 갑신정변이 실패한 이유는 무엇인가요?
------------------------------------------------------------


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


답변:
갑신정변이 실패한 원인 중 하나는 일본과의 협력 문제 때문이었습니다. 당시 조선에서는 일본이라는 외세를 이용하여 자신의 목표를 이루려고 했습니다만, 실제로는 효과적이지 않았어요. 또한, 국내에서의 신분제도와 정치체계 등 여러 가지 문제가 겹쳐져 있었는데, 특히 경영 효율성을 높이고 세금이나 군비 확보를 위해 필요한 기본적인 사회경제적 기반이 미흡했던 것이 큰 약점이 되기도 했죠. 이런 상황들이 결국 정변 자체가 무산되게 만들었습니다.

질문: 1894년 농민전쟁의 원인과 성격은 무엇인가요?
------------------------------------------------------------
답변:
1894년 농민전쟁의 원인이 된 것은 크게 두 가지 요소가 있었습니다. 먼저, 지방에서 일어난 부패한 행정과 고통스러운 세금 짓음 때문에 많은 사람들이 분노하게 됐습니다. 그리고 다른 중요한、동학 운동이 비록 사상을 가지고 있지만, 동시에 민중 운동의 기반 역할을 해냈다는 것입니다. 

농민전쟁은 단순히 농의 불만을 표현하는 것뿐만 아니라, 외세에 대한 저항까지 포함된 복잡한 의미를 가졌습니다. 그래서 우리는 이 전쟁이 그냥 반봉건적인 행동이 아니었다고 생각해야 합니다. 그것은 때로는 외세로부터 자유롭다는 의지를 나타내며, 때로는 본토의 잘못된 관행을 고치라는 메시지도 담겨있었습니다. 따라서 이 전쟁은 우리가 오늘날처럼 깊숙이 들어간 근대적 국가 건설의 시작이라고 할 수도 있습니다.


## 8. 대화형 테스트 (선택)

직접 입력해서 챗봇과 대화해볼 수 있습니다.  
종료하려면 `quit` 또는 `종료`를 입력하세요.

In [17]:
# 대화형 테스트 (Jupyter 환경에서 직접 입력)
print("한국 근대사 챗봇과 대화를 시작합니다. ('quit' 또는 '종료' 입력 시 종료)")
print("="*60)

interactive_history = []

while True:
    user_input = input("\n질문: ").strip()
    
    if not user_input:
        continue
    if user_input.lower() in ['quit', '종료', 'exit']:
        print("대화를 종료합니다.")
        break
    
    response = generate_history_response(
        user_query=user_input,
        chat_history=interactive_history,
        sections=history_sections,
        pipe=qwen_pipe
    )
    
    print(f"\n챗봇: {response}")
    
    interactive_history.append({"role": "user", "content": user_input})
    interactive_history.append({"role": "bot", "content": response})

한국 근대사 챗봇과 대화를 시작합니다. ('quit' 또는 '종료' 입력 시 종료)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



챗봇: 을사조약이라는 것이 있어요, 이건 우리 나라가 일본에 의해 강제로 맺게 된 악법이라고 할 수 있습니다. 그때부터 우리가 독립해서 자기 나라를 다스리는 게 불가능해졌어요. 이런 악법 때문에 우리의 자유와 권리를 잃게 되었구먼. 이를 통해 일본은 우리 나라의 주권을 상실하도록 만들었습니다. 그래서 이 조약은 우리 나라에서 큰 아픔을 가져왔죠.


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



챗봇: 을사조약은 1894년 10월 10일에 체결되었습니다. 이는 일명 ‘백두산 사건’ 이후인 해방 후 두달여 만의 일이예요.


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



챗봇: 을사조약은 1894년 10월 10일에 만들어진 거예요. 이 날짜는 백두산 사건이 발생한 지 두 달 정도 후였습니다.


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



챗봇: 백두산 사건이라면, 이거야. 1894년 초순에 있었던 일인데, 당시 우리는 일본과 갑작스럽게 식전전쟁을 벌이고 있었습니다. 그런데 어느날 갑자기 일본 군함들이 백두산 지역까지 들어오면서 무력 충돌이 생겼어요. 그렇게 인근 마을 사람들이 피해를 입거나 목숨을 잃기도 했습니다. 이것이 백두산 사건이란 이름이 붙은 이유겠죠. 이 사건 덕분에 억압적인 통치가 시작되었고, 결국에는 을사조약 같은 악법들을 맺어야 하는 계기가 됐어요.
대화를 종료합니다.


## 9. history_logic.py 파일 생성

프랑스어 챗봇의 `french_logic.py`와 동일한 패턴으로 `history_logic.py`를 생성합니다.  
이 파일을 NewLearn 프로젝트 `app.py`와 같은 폴더에 넣으면 역사 챗봇이 활성화됩니다.

### app.py에서 사용 방법
```python
from history_logic import get_history_bot_result

# call_llm 함수 내 역사 과목 분기에서:
if subject == "역사":
    response = get_history_bot_result(last_user_message, history)
```

In [18]:
# history_logic.py 파일 내용 정의
HISTORY_LOGIC_CODE = '''
"""
history_logic.py
한국 근대사 교육 챗봇 로직 모듈

NewLearn app.py에서 다음과 같이 임포트해서 사용합니다:
    from history_logic import get_history_bot_result

call_llm() 함수 내 역사(歷史) 과목 분기에 연결하면 됩니다.
"""

import os
import re
import torch
from transformers import pipeline
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ─── 설정 ───────────────────────────────────────────────────────────────────
MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"          # 메모리에 맞게 변경 가능
DATA_FILE  = "korean_modern_history_chatbot_ready.txt"  # 학습 데이터 파일 경로
TOP_K      = 3                                      # 검색할 상위 섹션 수
MAX_TOKENS = 512                                    # 최대 생성 토큰
# ────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """당신은 한국 근대사 전문 교육 챗봇입니다.
아래 [참고 자료]를 바탕으로 학생의 질문에 친절하고 정확하게 답변하세요.

응답 규칙:
응답 규칙:
1. 반드시 한국어(한글)로만 답변합니다.
2. 한자를 절대 출력하지 않습니다. 괄호 안에도 한자 금지입니다.
3. [참고 자료]에 있는 내용만을 바탕으로 답변합니다. 자료에 없는 내용은 추가하지 않습니다.
4. 모르는 내용은 '이 부분은 제 자료에 없습니다'라고 말합니다.
5. 중학생이 이해하기 쉽게 기 - 승 - 전 - 결 순서로 설명합니다.
6. 어려운 역사 용어는 바로 뒤에 쉬운 말로 풀어 설명합니다. 예: "통리기무아문 — 외교·군사 담당 기관"
7. 선생님이 학생에게 이야기하는 말투로 작성합니다.
8. 4문장 이상으로 답변합니다."""


# ─── 모델 & 데이터 지연 로드 (처음 호출 시 한 번만 초기화) ─────────────────
_pipe     = None
_sections = None


def _remove_hanja(text):
    """생성된 텍스트에서 한자(CJK)를 제거합니다."""
    import re as _re
    return _re.sub(r"[一-鿿㐀-䶿豈-﫿]+", "", text)


def _load_resources():
    """모델과 역사 데이터를 처음 사용 시 한 번만 로드합니다."""
    global _pipe, _sections

    if _sections is None:
        _sections = _load_history_data(DATA_FILE)

    if _pipe is None:
        device = 0 if torch.cuda.is_available() else -1
        dtype  = torch.float16 if device == 0 else torch.float32
        _pipe  = pipeline(
            task="text-generation",
            model=MODEL_ID,
            device=device,
            trust_remote_code=True,
            torch_dtype=dtype,
        )


def _load_history_data(file_path: str) -> list:
    """한국근대사 텍스트 파일을 읽어 섹션 단위 리스트로 반환합니다."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()
    except FileNotFoundError:
        return []

    sections = re.split(r"(?=###)", raw_text)
    parsed = []
    for section in sections:
        section = section.strip()
        if not section or not section.startswith("#"):
            continue

        title_match = re.match(r"#{2,3}\\s+(.+)", section)
        title = title_match.group(1).strip() if title_match else "(제목 없음)"

        kw_match = re.search(r"주요 키워드[:\\s]+(.+)", section)
        keywords = [k.strip() for k in kw_match.group(1).split(",")] if kw_match else []

        q_match = re.search(r"예상 질문[:\\s]+(.+)", section)
        questions = [q.strip() for q in q_match.group(1).split("/")] if q_match else []

        parsed.append({
            "title": title,
            "content": section,
            "keywords": keywords,
            "questions": questions,
        })
    return parsed


def _retrieve_context(user_query: str, sections: list, top_k: int = 2) -> str:
    """사용자 질문과 가장 관련 있는 섹션을 검색하여 컨텍스트로 반환합니다."""
    query_tokens = set(re.findall(r"[가-힣a-zA-Z0-9]{2,}", user_query))

    scores = []
    for sec in sections:
        title_tokens = set(re.findall(r"[가-힣a-zA-Z0-9]{2,}", sec["title"]))
        kw_tokens    = set(re.findall(r"[가-힣a-zA-Z0-9]{2,}", " ".join(sec["keywords"])))
        q_tokens     = set(re.findall(r"[가-힣a-zA-Z0-9]{2,}", " ".join(sec["questions"])))

        score = (
            len(query_tokens & title_tokens) * 3
            + len(query_tokens & kw_tokens) * 2
            + len(query_tokens & q_tokens) * 1
        )
        scores.append((score, sec))

    scores.sort(key=lambda x: x[0], reverse=True)
    top_sections = [sec for score, sec in scores[:top_k] if score > 0]

    if not top_sections:
        return "이 챗봇은 한국 근대사(1860년대~1945년) 전반을 다룹니다."

    parts = []
    for sec in top_sections:
        core_match = re.search(
            r"핵심 포인트:[\\s\\S]+?(?=주요 키워드|예상 질문|$)", sec["content"]
        )
        core_text = core_match.group(0).strip() if core_match else sec["content"][:300]
        parts.append(f"[{sec[\'title\']}]\\n{core_text}")

    return "\\n\\n".join(parts)


# ─── 공개 API ────────────────────────────────────────────────────────────────

def get_history_bot_result(user_query: str, chat_history: list = None) -> str:
    """
    한국 근대사 질문에 대한 QWEN 모델의 답변을 반환합니다.

    Args:
        user_query  : 사용자 질문 문자열
        chat_history: app.py histories 형식 [{"role": "bot"/"user", "content": ..., "time": ...}, ...]

    Returns:
        str: 모델 응답 텍스트 (HTML 태그 없는 순수 텍스트)
    """
    if chat_history is None:
        chat_history = []

    # 처음 호출 시 모델 & 데이터 로드
    _load_resources()

    # RAG 컨텍스트 검색
    context = _retrieve_context(user_query, _sections, top_k=TOP_K)

    # 메시지 구성
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # 최근 대화 이력 (최대 8턴)
    for turn in chat_history[-8:]:
        role    = "assistant" if turn.get("role") == "bot" else "user"
        content = re.sub(r"<[^>]+>", "", turn.get("content", ""))  # HTML 태그 제거
        messages.append({"role": role, "content": content})

    messages.append({
        "role": "user",
        "content": f"[참고 자료]\\n{context}\\n\\n[질문]\\n{user_query}",
    })

    # 모델 추론
    outputs = _pipe(
        messages,
        max_new_tokens=MAX_TOKENS,
        temperature=0.3,
        do_sample=True,
        repetition_penalty=1.2,
        return_full_text=False,
    )

    generated = outputs[0]["generated_text"]

    # 리스트 형식 처리 (일부 모델 버전에서 다르게 반환)
    if isinstance(generated, list):
        for msg in reversed(generated):
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                _remove_hanja(msg.get("content", "").strip())
        return _remove_hanja(str(generated[-1]).strip())

    return _remove_hanja(str(generated).strip())
'''


# history_logic.py 저장 경로
# 이 파일을 NewLearn 프로젝트 루트(app.py와 같은 폴더)에 복사해 주세요.
OUTPUT_PATH = "history_logic.py"

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    f.write(HISTORY_LOGIC_CODE.strip())

print(f"history_logic.py 생성 완료: {OUTPUT_PATH}")
print("\n이 파일을 NewLearn 프로젝트 폴더(app.py와 같은 위치)에 복사하면 됩니다.")
print("\n[app.py 수정 방법]")
print("1. 파일 상단에 추가:")
print('   from history_logic import get_history_bot_result')
print("\n2. call_llm() 함수 내 역사 과목 분기 추가:")
print('   if subject == "역사":')  
print('       return get_history_bot_result(last_message, history)')

history_logic.py 생성 완료: history_logic.py

이 파일을 NewLearn 프로젝트 폴더(app.py와 같은 위치)에 복사하면 됩니다.

[app.py 수정 방법]
1. 파일 상단에 추가:
   from history_logic import get_history_bot_result

2. call_llm() 함수 내 역사 과목 분기 추가:
   if subject == "역사":
       return get_history_bot_result(last_message, history)


## 10. app.py 병합 가이드

생성된 `history_logic.py`를 NewLearn 프로젝트에 연결하는 방법입니다.

### app.py 수정 사항

**① 상단 임포트 추가**
```python
from history_logic import get_history_bot_result
```

**② `call_llm()` 함수 교체**
```python
def call_llm(subject, history):
    """과목별로 LLM 로직을 분기합니다."""
    if subject == "역사":
        return get_history_bot_result(
            user_query=history[-1]["content"],
            chat_history=history[:-1]   # 마지막 사용자 메시지 제외한 이력
        )
    
    # 다른 과목은 기존 로직 유지
    last = history[-1]["content"]
    short = f'{last[:40]}{"..." if len(last) > 40 else ""}'
    return f'"{short}"에 대한 답변입니다.<br>{subject} 맥락에 맞춰 LLM이 응답합니다.'
```

### 주의사항
- `korean_modern_history_chatbot_ready.txt`도 함께 같은 폴더에 업로드해야 합니다.
- `requirements.txt`에 `transformers`, `torch`, `accelerate` 추가가 필요합니다.
- Streamlit Cloud에서는 GPU가 없으므로 CPU 모드로 동작하며, 응답 속도가 느릴 수 있습니다.